In [42]:
%pylab inline 
import torch 
import torch.nn as nn 
import torch.nn.functional as F 

from torchtext.datasets import WikiText2
from torchtext.data.utils import get_tokenizer
from collections import Counter
from torchtext.vocab import Vocab

from tqdm import trange

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


Populating the interactive namespace from numpy and matplotlib


In [35]:
train_iter = WikiText2(split="train")
tokenizer = get_tokenizer("basic_english")
counter = Counter()
for line in train_iter:
    counter.update(tokenizer(line))
vocab = Vocab(counter)
# turns text to tensor 
def data_process(raw_text_iter):
    data = [torch.tensor([vocab[token] for token in tokenizer(item)], 
                          dtype=torch.long) for item in raw_text_iter]
    return torch.cat(tuple(filter(lambda t: t.numel() > 0, data)))

def batchify(data, bsz):
    n_batch = data.size(0) // bsz
    data = data.narrow(0, 0, n_batch * bsz)
    data = data.view(bsz, -1).t().contiguous()
    return data.to(device)
# Loading data into tensors
train_iter, val_iter, test_iter = WikiText2()
train_data = data_process(train_iter)
val_data = data_process(val_iter)
test_data = data_process(test_iter)

# batching data 
batch_size = 20
test_batch_size = 10
train_data = batchify(train_data, batch_size)
val_data = batchify(val_data, test_batch_size)
test_data = batchify(test_data, test_batch_size)



In [36]:
# Generate input and target sequence 
bptt = 35 
def get_batch(source, i):
    seq_len = min(bptt, len(source) -1 - i)
    data = source[i:i+seq_len]
    target = source[i+1:i+1+seq_len].reshape(-1)
    return data, target

